# Stream arm state

Continuously print joint and wrist state in teach mode.
This mirrors `bot01/sdk/cli/stream_arm_state.py`.
If `bot01` is available, a relative wrist pose is also printed.

Interrupt the kernel to stop the loop.


In [ ]:
import time

import numpy as np

from r2_labs import client, rpc_api

try:
  from bot01.dm_robotics.geometry import geometry
except ImportError:
  geometry = None

np.set_printoptions(precision=4, floatmode="fixed", suppress=True, linewidth=100)

host = "localhost"
server_address = f"tcp://{host}:{rpc_api.DEFAULT_PORT}"
query_address = f"tcp://{host}:{rpc_api.DEFAULT_QUERY_PORT}"

robot = client.Robot(server_address, query_server_address=query_address)


In [ ]:
robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.TEACH)
first_wrist_pose = None

while True:
  proprio = robot.raw_robot.get_proprio_data()

  print(f"
JOINT POSITIONS: {proprio.joint_positions}")
  print(f"
JOINT VELOCITIES: {proprio.joint_velocities}")
  print(f"GRIPPER POSITION: {proprio.gripper_positions}")
  print(f"WRIST POSE: {proprio.wrist_pose}")

  if geometry is not None and proprio.wrist_pose is not None:
    if first_wrist_pose is None:
      first_wrist_pose = proprio.wrist_pose

    initial_pose = geometry.PoseStamped(
        geometry.Pose(first_wrist_pose[:3], first_wrist_pose[3:])
    )
    cur_pose = geometry.PoseStamped(
        geometry.Pose(proprio.wrist_pose[:3], proprio.wrist_pose[3:])
    )

    rel_pose = geometry.frame_relative_pose(cur_pose, initial_pose)
    print(f"REL WRIST POSE: {rel_pose}")

  time.sleep(1.0)
